In [12]:
# pip install sklearn
# pip install pandas
# pip install six
# pip install pydotplus
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

from sklearn import metrics # is used to create classification results
from sklearn.tree import export_graphviz # is used for plotting the decision tree
from six import StringIO # is used for plotting the decision tree
from IPython.display import Image # is used for plotting the decision tree
from IPython.core.display import HTML # is used for showing the confusion matrix
import pydotplus # is used for plotting the decision tree
from numbers import Number

# Set the classifier, this can be Label or attack_cat
classifier = 'attack_cat'
# Data set already comes with column names!
col_names = ['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','service','Sload','Dload','Spkts','Dpkts','swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Stime','Ltime','Sintpkt','Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ ltm','ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','attack_cat','Label']
# Because we are taking the column names from the data set, we can take advantage of pandas' default behaviors!
# col_names isn't necessary TO READ THE CSV FILE! We may need to use it to factorize the data!
# header set to infer, which without column names means 0, aka the first line of the data set
# skiprows is none by default
pima_raw = pd.read_csv("../UNSW-NB15-BALANCED-TRAIN.csv")
#factorize pima raw into pima
pima = pima_raw
pima_enum = []
classifier_uniques = []
for col_n in col_names:
    if ((isinstance(pima_raw[col_n][0], Number) == False) or pd.isna(pima_raw[col_n][0])):
        temp_codes, temp_uniques = pd.factorize(pima_raw[col_n])
        pima[col_n] = temp_codes
        pima_enum.append(temp_uniques)
        # if the column name is attack_cat, keep that column for the display
        if(col_n == classifier):
            classifier_uniques = temp_uniques
        
#if this doesn't work, comment out the conditional statement

# Get first 5k rows for training data
training_data = pima[:20000]
# Get next 1k rows for validation data
validation_data = pima[20000:30000]
# Get next 1k rows for testing data
testing_data = pima[30000:40000]

# Set the feature columns.
feature_cols =['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','service','Sload','Dload','Spkts','Dpkts','swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Stime','Ltime','Sintpkt','Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ ltm','ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm']
# label_cols =['attack_cat', 'Label'] # THIS DOES NOT WORK!!!
# Get the feature columns from all data sets.
training_data_x = training_data[feature_cols]
validation_data_x = validation_data[feature_cols]
testing_data_x = testing_data[feature_cols]
# Get the target label from all data sets.
training_data_y = training_data[classifier]
validation_data_y = validation_data[classifier]
testing_data_y = testing_data[classifier]

# Creating the decision tree classifier object
clf = DecisionTreeClassifier(criterion='entropy')
clf = clf.fit(training_data_x,training_data_y)

# Use the model to predict y values
prediction_data_y = clf.predict(validation_data_x)

# Compare to validation data
print("Accuracy: {:.2f}%\n".format(metrics.accuracy_score(validation_data_y, prediction_data_y)*100))
# display(HTML(pd.DataFrame(metrics.confusion_matrix(validation_data_y, prediction_data_y), columns=[classifier_uniques], index=[classifier_uniques]).to_html()))
print(metrics.classification_report(validation_data_y, prediction_data_y))

Accuracy: 92.26%

              precision    recall  f1-score   support

          -1       0.99      0.98      0.99      4999
           0       0.98      0.98      0.98      3410
           1       0.59      0.84      0.70        82
           2       0.64      0.64      0.64       683
           3       0.29      0.35      0.32       234
           4       0.74      0.76      0.75       280
           5       0.75      0.74      0.74       182
           6       0.22      0.06      0.10        31
           7       0.93      0.82      0.88        17
           8       0.19      0.15      0.17        41
           9       0.33      0.33      0.33         3
          10       0.54      0.72      0.62        18
          11       0.00      0.00      0.00        13
          12       0.50      0.29      0.36         7

    accuracy                           0.92     10000
   macro avg       0.55      0.55      0.54     10000
weighted avg       0.92      0.92      0.92     10000

